# 目的<font color="red">【没】</font>
* ~~MongoDBに登録したJalcのメタデータをApache Solrに登録するにCSVファイルに書き出す~~
* ~~スキーマ：<font color = "Red">筆頭著者名, タイトル, 出版年, 雑誌名, 巻号, 項番号</font>~~
* CSVファイルにせず，登録する方法があったため，当該ファイルは没となった

## <font color="Red">要実行</font>

In [1]:
#モジュール及び接続文字列の読み込み
import pymongo
url_mongodb = "mongodb://localhost:27017/"

#mongodbへの接続
myclient = pymongo.MongoClient(url_mongodb)
mydb = myclient["jalc"]
mycol = mydb["restapi"]

## 事前確認
* 実行しなくても良い

### タイトル属性について

In [ ]:
#タイトル属性を持たないドキュメントを確認
num = mycol.count_documents({"title_list":{"$exists":False}})
print(f"title_listを持たないドキュメント：{num}件")

title_listを持たないドキュメント：517件


In [3]:
#title_list.langを持たないドキュメントを確認
num = mycol.count_documents({"title_list.lang":{"$exists":False}})
print(f"title_list.langを持たないドキュメント：{num}件")

title_list.langを持たないドキュメント：3961834件


In [8]:
#タイトルの言語を確認
title_languages = []

#クエリ
query = {"$and":[{"title_list":{"$exists":True}},{"title_list.lang":{"$exists":True}}]}

for doc in mycol.find(query):
    title_list = doc["title_list"]
    for val in title_list:
        #print(val['lang'])
        lang = val.get("lang","none")
        if lang != "none" and lang not in title_languages:
            title_languages.append(lang)

#標準出力
print(f"言語数：{len(title_languages)}")
print(title_languages)             

言語数：65
['en', 'ja', 'de', 'fr', 'la', 'es', 'ru', 'pl', 'zh', 'ko', 'it', 'co', 'cs', 'id', 'fi', 'sk', 'ms', 'lb', 'no', 'jv', 'sv', 'eu', 'da', 'nl', 'ro', 'pt', 'el', 'hi', 'bs', 'sw', 'bg', 'sn', 'uz', 'mi', 'af', 'is', 'mr', 'lt', 'gu', 'bn', 'ca', 'yo', 'hr', 'sr', 'ny', 'te', 'kn', 'ht', 'gd', 'hu', 'vi', 'lo', 'ta', 'si', 'km', 'eo', 'tr', 'th', 'fa', 'ur', 'my', 'mn', 'ar', 'am', 'bo']


In [11]:
#title_list.lang属性を持たないドキュメントを確認
count_lis = []
query = {"$and":[
    {"title_list":{"$exists":True}},
    {"title_list.lang":{"$exists":False}}
]}

for doc in mycol.find(query):
    num = len(doc["title_list"])
    if num not in count_lis:
        count_lis.append(num)

print(count_lis)        

[1]


In [16]:
#title_list配下の属性を確認
attributes = []
query = {"title_list":{"$exists":True}}

for doc in mycol.find(query):
    title_list = doc.get("title_list")
    for x in title_list:
        for attribute in x.keys():
            if attribute not in attributes:
                attributes.append(attribute)

print(attributes)                

['lang', 'title', 'subtitle']


In [18]:
#タイトルに日本語・英語の両方を使わず, 2言語以上の表記がされている論文の確認
count = []
query = {"$and":[
    {"title_list.lang":{"$exists":True}},
    {"title_list.lang":{"$ne":"ja"}},
    {"title_list.lang":{"$ne":"en"}}
]}

for doc in mycol.find(query):
    title_list = doc["title_list"]
    if len(title_list) not in count:
        count.append(len(title_list))

print(count)        

[1, 2]


In [3]:
# title_list配下に格納されているオブジェクト数が３のドキュメント数を確認
query = {
    "title_list.2":{"$exists":True}
}

num = mycol.count_documents(query)
print(f"該当件数：{num}件")

該当件数：3827件


In [9]:
# title_list配下のオブジェクト数が3の言語を確認
query = {
    "title_list.2":{"$exists":True}
}

projection = {
    "_id":0,
    "title_list":1
}

languages = []

for doc in mycol.find(query,projection):
    tmp = []
    for x in doc["title_list"]:
        lang = x.get("lang","none")
        tmp.append(lang)
    tmp = sorted(tmp)
    if tmp not in languages:
        languages.append(tmp)    
        
for val in languages:
    print(val)

['en', 'ja', 'none']
['en', 'ja', 'zh']
['de', 'en', 'ja']
['en', 'es', 'ja']
['en', 'ja', 'ru']
['en', 'ja', 'mn']
['en', 'fr', 'ja']
['en', 'ja', 'th']


### クリエーター属性について

In [43]:
#creator_list配下の属性を確認
attributes = []
query = {"creator_list":{"$exists":True}}

for doc in mycol.find(query):
    title_list = doc.get("creator_list")
    for x in title_list:
        for attribute in x.keys():
            if attribute not in attributes:
                attributes.append(attribute)

print(attributes)  

['sequence', 'type', 'names', 'affiliation_list', 'researcher_id_list']


In [54]:
#クリエーター名の言語を確認
languages = []
query = {"$and":
         [
             {"creator_list.names.lang":{"$exists":True}}
         ]
         }

for doc in mycol.find(query,{"_id":0,"creator_list.names.lang":1,"url":1}):
    #print(doc["creator_list"][0]["names"])
    for x in doc["creator_list"][0]["names"]:
        #print(x["lang"])
        try:
            if x["lang"] not in languages:
                languages.append(x["lang"])
        except:
            print(doc["url"])
            break
print(languages)

https://doi.org/10.11179/ker1926.20.2_1
https://doi.org/10.11179/ker1926.20.2_39
https://doi.org/10.11179/ker1926.57.2_29
https://doi.org/10.11179/ker1926.57.2_40
https://doi.org/10.50870/00000321
https://doi.org/10.50870/00000322
https://doi.org/10.50870/00000323
https://doi.org/10.50870/00000325
https://doi.org/10.50870/00000326
https://doi.org/10.50870/00000327
https://doi.org/10.50870/00000328
https://doi.org/10.50870/00000329
https://doi.org/10.50870/00000330
https://doi.org/10.50870/00000331
https://doi.org/10.50870/00000332
https://doi.org/10.50870/00000333
https://doi.org/10.50870/00000334
https://doi.org/10.50870/00000335
https://doi.org/10.50870/00000336
https://doi.org/10.50870/00000337
https://doi.org/10.50870/00000338
https://doi.org/10.50870/00000339
https://doi.org/10.50870/00000340
https://doi.org/10.50870/00000342
https://doi.org/10.50870/00000343
https://doi.org/10.50870/00000344
https://doi.org/10.50870/00000345
https://doi.org/10.50870/00000346
https://doi.org/10.50

In [123]:
#langを持たないものを確認
query = {
    "$and":[
        {"creator_list":{"$exists":True}},
        {"creator_list.names.lang":{"$exists":False}}
    ]
}

print(mycol.count_documents(query))

4307513


In [ ]:
#クリエータ名が日本語であるか判定
import re

def has_japanese(text):
    """
    ・引用文献文字列に日本語が含まれるか否かの判定
    ・unicodeを利用
    ひらがな：u3040-u309F
    カタカナ：u30A0-u30FF
    半角カナ：uFF66-uFF9f
    一般漢字：u4E00-u9FFF
    """
    return bool(re.search(r'[\u3040-\u309F\u30A0-\u30FF\uFF66-\uFF9F\u4E00-\u9FFF]', text))

query = {"url":"https://doi.org/10.24482/00000071"}
for doc in mycol.find(query):
    print(doc["creator_list"])
    for x in doc["creator_list"]:
        #print(x["names"])
        print(x["names"][0]["last_name"],x["names"][0]["first_name"],has_japanese(x["names"][0]["first_name"]))


[{'sequence': '1', 'type': 'person', 'names': [{'last_name': '村田', 'first_name': '浩子'}]}, {'sequence': '2', 'type': 'person', 'names': [{'lang': 'en', 'last_name': 'Hiroko', 'first_name': 'MURATA'}]}]
村田 村田 True
Hiroko Hiroko False


In [15]:
query = {
    "creator_list":{"$exists":True}
}

languages = []

for doc in mycol.find(query):
    creator_list = doc.get("creator_list","")
    flag = False
    tmp1 = []
    for x in creator_list:
        tmp2 = []
        if len(x["names"]) == 3:
            flag = True
        for y in x["names"]:
            lang = y.get("lang","none")
            tmp2.append(lang)
        tmp1.append(sorted(tmp2))
    if flag:
        languages.append(tmp1)        
            

for z in languages:
    print(z)

[['en', 'ja', 'zh']]
[['en', 'ja', 'zh']]
[['en', 'ja', 'zh']]
[['en', 'ja', 'zh']]
[['en', 'ja', 'zh']]
[['en', 'ja', 'zh']]
[['en', 'ja', 'zh']]
[['en', 'ja', 'zh']]
[['en', 'ja', 'zh']]
[['en', 'ja', 'zh']]
[['en', 'ja', 'zh']]
[['en', 'ja', 'zh']]
[['en', 'ja', 'zh']]
[['en', 'ja'], ['en', 'ja'], ['en', 'ja', 'zh'], ['en', 'ja']]
[['en', 'ja'], ['en', 'ja'], ['en', 'ja', 'zh'], ['en', 'ja'], ['en', 'ja']]
[['en', 'ja', 'zh'], ['en', 'ja'], ['en', 'ja']]
[['en', 'ja', 'zh']]
[['en', 'ja', 'zh']]
[['en', 'ja', 'zh']]
[['en', 'ja', 'zh']]
[['en', 'ja', 'zh']]
[['en', 'ja', 'zh']]
[['en', 'ja', 'zh']]
[['en', 'ja'], ['en', 'ja', 'none']]
[['en', 'ja', 'none']]
[['en', 'ja', 'none']]
[['en', 'ja', 'none']]
[['en', 'ja', 'none']]
[['en', 'ja', 'none'], ['en', 'ja'], ['en', 'ja']]
[['en', 'ja', 'none']]
[['en', 'ja', 'none']]
[['en', 'ja', 'none']]
[['en', 'ja', 'ru']]
[['en', 'ja', 'none']]
[['en', 'ja', 'none']]
[['en', 'ja', 'none']]
[['en', 'ja', 'ru']]
[['en', 'ja', 'none']]
[['en', 

### ジャーナル属性について

In [15]:
# type属性の有無
query = {
       "$and":[
              {"journal_title_name_list":{"$exists":True}},
              {"journal_title_name_list.3":{"$exists":True}}
       ]
}

lang_list = []

for doc in mycol.find(query):    
       journal_title_name_list = doc.get("journal_title_name_list","")
       tmp = []
       full_count = 0
       for val in journal_title_name_list:
              lang = val.get("lang","none")
              Type = val.get("type","")
              tmp.append(lang)
              if Type == "full":
                     full_count += 1
       if full_count == 3:
              print(doc["url"])  
              break                  

https://doi.org/10.2151/jmsj1923.17.5_208


In [2]:
query = {
    "journal_title_name_list":{"$exists":True}
}

List = []

for doc in mycol.find(query):
    journal_title_name_list = doc.get("journal_title_name_list","")
    hasType = True
    Size = len(journal_title_name_list)
    for val in journal_title_name_list:
        Type = val.get("type","none")
        if Type == "none":
            hasType = False
    if not hasType and Size not in List:
        List.append(Size)

print(List)        


[1]


In [11]:
langs = []
query = {
    "journal_title_name_list":{"$exists":True}
}
for doc in mycol.find(query):
    journal_title_name_list = doc.get("journal_title_name_list","")
    tmp = []
    for val in journal_title_name_list:
        lang = val.get("lang","none")
        Type = val.get("type","")
        if Type == "full":
            tmp.append(lang)
    tmp = sorted(tmp)
    if tmp not in langs:
        langs.append(tmp)        

for r in langs:
    print(r)


['en', 'ja', 'none']
['en']
['en', 'none']
['en', 'ja']
['en', 'ja', 'ja']
[]
['en', 'ja', 'ja', 'none']
['ja']
['none']


## CSVへの書き出し

In [ ]:
# 全件ドキュメント
documents = mycol.find()


In [2]:
# CSVファイルの作成
import re

# 文字列が日本語であるか否か
def has_japanese(text):
    return bool(re.search(r'[\u3040-\u309F\u30A0-\u30FF\uFF66-\uFF9F\u4E00-\u9FFF]', text))

# 名前の結合
def concat_name(first_name,last_name,lang):
    name = ""

    # langを持たない場合
    if lang == "":
        if has_japanese(first_name):
            lang = "ja"
        else:
            lang = "en"

    # 結合処理
    if lang in ["ja","zh","ko"]:
        if last_name == "":
            name = first_name
        else:
            name = last_name + " " + first_name
    else:
        if last_name == "":
            name = first_name
        else:
            name = first_name + " " + last_name

    return name                        

# ドキュメントからタイトルを抽出
def get_title(doc):
    title1,title2,title3,title4 = "","","",""

    title_list = doc.get("title_list","")

    if title_list == "":
        return title1,title2,title3,title4
    else:
        if len(title_list) == 1:
            title1,title2 = title_list[0]["title"],title_list[0]["title"]
            subtitle = title_list[0].get("subtitle","")
            if subtitle != "":
                title2 = title2 + " " + subtitle
            return title1,title2,title3,title4
        elif len(title2) == 2:
            title1,title2,title3,title4 = title_list[0]["title"],title_list[0]["title"],title_list[1]["title"],title_list[1]["title"]
            subtitle1,subtitle2 = title_list[0].get("subtitle",""),title_list[1].get("subtitle","")
            if subtitle1 != "":
                title2 = title2 + " " + subtitle1
            if subtitle2 != "":
                title4 = title4 + " " + subtitle2
            return title1,title2,title3,title4
        else:
            isFirst = True
            for i in range(len(title_list)):
                lang = title_list[i].get("lang","")
                if lang in ["ja","en"]:
                    if isFirst:
                        isFirst = False
                        title1,title2 = title_list[i]["title"],title_list[i]["title"]
                        subtitle1 = title_list[i].get("subtitle","")
                        if subtitle1 != "":
                            title2 = title2 + " " + subtitle1
                    else:
                        title3,title4 = title_list[i]["title"],title_list[i]["title"]
                        subtitle2 = title_list[i].get("subtitle","")
                        if subtitle2 != "":
                            title4 = title4 + " " + subtitle2
            return title1,title2,title3,title4                        
                    
#ドキュメントからクリエーターを抽出
def get_creator(doc):
    creator1,creator2 = "",""
    creator_list = doc.get("creator_list","")

    if creator_list == "":
        return creator1,creator2
    else:
        # クリエータ名をリストで保存
        tmp1,tmp2 = [],[]
        # 分類基準言語
        first_lang,second_lang = "",""

        for val in creator_list:    # 各著者について
            if len(val["names"]) == 1:  # 表記が1通りの場合
                first_name,last_name = val["names"][0].get("first_name",""),val["names"][0].get("last_name","")
                lang = val["names"][0].get("lang","")
                name = concat_name(first_name,last_name,lang)
                tmp1.append(name)
                tmp2.append(name)
            elif len(val["names"]) == 2: # 表記が2通りの場合
                for x in val["names"]: # 各表記について
                    first_name,last_name,lang = x.get("first_name",""),x.get("last_name",""),x.get("lang","")

                    # langを持たない場合
                    if lang == "":
                        if has_japanese(first_name):
                            lang = "ja"
                        else:
                            lang = "en"

                    name = concat_name(first_name,last_name,lang)        

                    if first_lang == "" and second_lang == "":
                        first_lang = lang
                        tmp1.append(name)
                    elif first_lang != "" and second_lang == "":
                        second_lang = lang                       
                        tmp2.append(name)
                    else:
                        if lang == first_lang:
                            tmp1.append(name)
                        if lang == second_lang:
                            tmp2.append(name)  

            else: # 表記が3通りの場合
                for x in val["names"]: # 各表記について
                    first_name,last_name,lang = x.get("first_name",""),x.get("last_name",""),x.get("lang","")
                    name = concat_name(first_name,last_name,lang)

                    if first_lang == "" and second_lang == "":
                        first_lang,second_lang = "ja","en"

                    if lang == first_lang:
                        tmp1.append(name)
                    if lang == second_lang:
                        tmp2.append(name)

        creator1 = (",").join(tmp1)
        creator2 = (",").join(tmp2)
        return creator1,creator2                        

#ドキュメントから出版年を取得
def get_year(doc):
    year = ""
    publication_date = doc.get("publication_date","")

    if publication_date == "":
        return year
    else:
        year = publication_date["publication_year"]
        return year

# ドキュメントからジャーナル名を抽出
def get_journal(doc):
    journal1,journal2 = "",""
    journal_title_name_list = doc.get("journal_title_name_list","")

    if journal_title_name_list == "":
        return journal1,journal2
    else:
        Size = len(journal_title_name_list) # 格納されている要素数
        if Size == 1:
            journal1 = journal_title_name_list[0].get("journal_title_name","")
            return journal1,journal2
        elif Size == 2:
            first_lang = ""
            # journal1について
            for val in journal_title_name_list:
                journal_title = val.get("journal_title_name","")
                Type = val.get("type","")
                lang = val.get("lang","")

                # lang属性を持たない場合
                if lang == "":
                    if has_japanese(journal_title):
                        lang = "ja"
                    else:
                        lang = "en"

                if Type == "full":
                    journal1 = journal_title
                    first_lang = lang
            # journal2 について        
            for val in journal_title_name_list:
                journal_title = val.get("journal_title_name","")
                lang = val.get("lang","")
                # lang属性を持たない場合
                if lang == "":
                    if has_japanese(journal_title):
                        lang = "ja"
                    else:
                        lang = "en"
                if lang != first_lang:
                    journal2 = journal_title            
            return journal1,journal2
        else:
            full_journal = []   # type==fullのもの
            not_full_journal = []   # type!=fullのもの
            first_lang = ""
            # typeによる分類
            for val in journal_title_name_list:
                journal_title = val.get("journal_title_name","")
                lang = val.get("lang","")
                Type = val.get("type","")
                # lang属性を持たない場合
                if lang == "":
                    if has_japanese(journal_title):
                        lang = "ja"
                    else:
                        lang = "en"
                if Type == "full":
                    full_journal.append([journal_title,lang,Type])
                else:
                    not_full_journal.append([journal_title,lang,Type])
            if len(full_journal) == 1:
                journal1 = full_journal[0][0]
                first_lang = full_journal[0][1]
                # journal2を取得（あれば）
                for x in not_full_journal:
                    if x[1] != first_lang:
                        journal2 = x[0]
                return journal1,journal2                
            else:
                for x in full_journal:
                    if first_lang == "":
                        journal1 = x[0]
                        first_lang = x[1]
                    else:
                        if x[1] != first_lang:
                            journal2 = x[0]
                            break
                return journal1,journal2            

# ドキュメントからページの範囲を取得
def get_page(doc):
    page = ""
    first_page = doc.get("first_page","")
    last_page = doc.get("last_page","")

    if first_page != "" and last_page != "":
        if first_page == last_page:
            page = first_page
        else:    
            page = first_page + " " + last_page
    else:
        page = first_page + last_page
    return page

# 巻号を取得
def get_volumeAndIssue(doc):
    vol_and_issue = ""
    volume = doc.get("volume","")
    issue = doc.get("issue","")

    if volume != "" and issue != "":
        vol_and_issue = volume + " " + issue
    else:
        vol_and_issue = volume + issue

    return vol_and_issue               

# 本処理

documents = mycol.find({"content_type":"JA"})
count = 0    
for doc in documents:
    count += 1
    try:
        #title1,title2,title3,title4 = get_title(doc)
        #creator1,creator2 = get_creator(doc)
        #year = get_year(doc)
        #journal1,journal2 = get_journal(doc)
        #volume = doc.get("volume","")
        #issue = doc.get("issue","")
        vol_and_issue = get_volumeAndIssue(doc)
        #page = get_page(doc)
        #doi = doc.get("doi","")
    except:
        print(f"エラー：{doc["url"]}")

    if count % 1000000 == 2:
        print(f"volume and issue : {vol_and_issue}")        


print(f"データ件数：{count}")    

volume and issue : 17 6
volume and issue : 62 5
volume and issue : 67 552
volume and issue : 36 4
volume and issue : 148
volume and issue : 
volume and issue : 
volume and issue : 37 3
データ件数：7895097


In [2]:
print(mycol.count_documents({"content_type":"JA"}))

7895097
